# Exploring coherence matrix spectrum estimation

We explore the replacing eigenvalue decomposition of coherence matrix in coherence analysis with values calculated using QR decomposition. We explore the computational efficiency, accuracy of this approach and explore justifications for why it might be working.

we first test on random data, then on real data from Brady's Hot Spring. This data can be downloaded via the [AWS S3 Explorer for the Open Energy Data Initiative](https://data.openei.org/s3_viewer?bucket=nrel-pds-porotomo&prefix=DAS%2FH5%2FDASH%2F). The data is stored in the `nrel-pds-porotomo` bucket and the `DAS/H5/DASH/` prefix. The specific files used in this notebook are:
- `PoroTomo_iDAS16043_160314083818.h5`
- `PoroTomo_iDAS16043_160314083848.h5`

In [ ]:
import time
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation
from sklearn.utils.extmath import randomized_svd

In [ ]:
def loadBradyHShdf5(file, normalize="yes"):
    """

    Parameters
    ----------
    file : str
        path to brady hotspring h5py data file
    normalize : str, optional
        "yes" or "no". Indicates whether or not to remove laser drift and
        normalize. The default is 'yes'.

    Returns
    -------
    data : np array
        channel by samples numpy array of data
    timestamp_arr : numpy array
        array of the timestamps corresponding to the various samples in the
        data. Timestamps for brady hotspring data are with respect to the
        beginning time of the survey.

    """
    with h5py.File(file, "r") as open_file:
        dataset = open_file["das"]
        time = open_file["t"]
        data = np.array(dataset)
        timestamp_arr = np.array(time)
    data = np.transpose(data)
    if normalize == "yes":
        nSamples = np.shape(data)[1]
        # get rid of laser drift
        med = np.median(data, axis=0)
        for i in range(nSamples):
            data[:, i] = data[:, i] - med[i]

        max_of_rows = abs(data[:, :]).sum(axis=1)
        data = data / max_of_rows[:, np.newaxis]
    return data, timestamp_arr


def windowed_spectra(
    data: np.array, subwindow_len: int, overlap, freq=None, sample_interval=1
):
    """
    Calculate the frequency domain representation of data in windows.
    """
    win_start = 0
    window_samples = int(subwindow_len / sample_interval)
    total_samples = data.shape[-1]
    overlap = int(overlap / sample_interval)
    intervals = np.arange(
        window_samples, total_samples + 1, window_samples, dtype=int
    )  # break time series into windowed intervals

    win_end = intervals[0]

    absolute_spectra = np.fft.rfft(data[:, win_start:win_end])
    win_spectra = absolute_spectra[np.newaxis]

    while win_end < total_samples:
        win_start = win_end - overlap
        win_end = win_start + window_samples
        absolute_spectra = np.fft.rfft(data[:, win_start:win_end])
        win_spectra = np.append(
            win_spectra, absolute_spectra[np.newaxis], axis=0
        )
        # win_start = win_end

    frequencies = np.fft.rfftfreq(window_samples, sample_interval)

    return win_spectra, frequencies


def normalised_windowed_spectra(
    data: np.array, subwindow_len: int, overlap, freq=None, sample_interval=1
):
    win_spectra, frequencies = windowed_spectra(
        data, subwindow_len, overlap, freq, sample_interval
    )

    # win_spectra = np.absolute(win_spectra)**2 # sub for next line
    # win_spectra = win_spectra * np.conjugate(win_spectra) # absolutes square of spectra. We need this if
    # we want to use the normalised spectra to calculate welch coherence.
    # normalizer = np.sum(win_spectra, axis=0)

    normalizer = np.sum(np.absolute(win_spectra) ** 2, axis=0)
    normalizer = np.tile(np.sqrt(normalizer), (win_spectra.shape[0], 1, 1))
    normalizer = normalizer.transpose(2, 1, 0)

    normalized_spectra = win_spectra.transpose(2, 1, 0) / normalizer

    return normalized_spectra, frequencies


def welch_coherence(
    data: np.array, subwindow_len: int, overlap, freq=None, sample_interval=1
):
    """
    Calculate the coherence matrix at all (or particular frequencies: yet to be implemented)
    using the welch method.
    """
    win_spectra, frequencies = windowed_spectra(
        data, subwindow_len, overlap, freq, sample_interval
    )

    normalizer = np.sum(np.absolute(win_spectra) ** 2, axis=0)
    normalizer = np.tile(normalizer, (normalizer.shape[0], 1, 1))
    normalizer = normalizer * normalizer.transpose((1, 0, 2))
    normalizer = normalizer.transpose(2, 1, 0)

    welch_numerator = np.matmul(
        win_spectra.transpose(2, 1, 0),
        np.conjugate(win_spectra.transpose(2, 0, 1)),
    )
    welch_numerator = np.absolute(welch_numerator) ** 2
    coherence = np.multiply(welch_numerator, 1 / normalizer)

    return coherence, frequencies

## Computational efficiency comparison

Compare the performance of the following methods for estimating the eigenvalue decomposition of the coherence matrix:
- Directly computing the eigenvalue decomposition of the coherence matrix.
- Using the QR decomposition of the coherence matrix to estimate the eigenvalue decomposition.
- Using the singular value decomposition of the coherence matrix to estimate the eigenvalue decomposition.
- Using the randomized singular value decomposition of the coherence matrix to estimate the eigenvalue decomposition.

In [ ]:
dim = 2000
dims = [100, 500, 1000, 2000, 3500, 5000, 7000, 9000, 10000]
qr_times = []
svd_times = []
eig_times = []
rsvd_times = []

for dim in dims:
    approx_rank = int(dim / 2)

    RandA = np.random.randn(dim, int(dim / 2))
    t0 = time.time()
    Q, R = np.linalg.qr(RandA)
    t1 = time.time()
    qr_time = t1 - t0

    # print("QR time: ", qr_time)
    qr_times.append(qr_time)

    t0 = time.time()
    U, S, Vh = np.linalg.svd(RandA)
    t1 = time.time()
    svd_time = t1 - t0

    # print("SVD time: ", svd_time)
    svd_times.append(svd_time)

    t0 = time.time()
    eigenvals, _ = np.linalg.eig(RandA @ RandA.transpose())
    t1 = time.time()
    eig_time = t1 - t0

    eig_times.append(eig_time)
    # print("Eigenvalue time: ", svd_time)

    t0 = time.time()
    rU, rS, rVh = randomized_svd(RandA, approx_rank)
    t1 = time.time()
    rsvd_time = t1 - t0

    rsvd_times.append(rsvd_time)
    # print("Randomized SVD time: ", rsvd_time)

Set up plotting parameters.

In [ ]:
fsize = 15
ticksize = 12
colors = ["#800000", "#FFD700", "#663399", "#000000", "#008B8B"]

### Plot timing results

In [ ]:
plt.plot(dims, eig_times, "-o", label="Eigenvalue", color="black")
plt.plot(dims, qr_times, "-^", label="QR", color="gold")
plt.plot(dims, svd_times, "-s", label="SVD", color=colors[2])
plt.plot(dims, rsvd_times, "-x", label="Randomized SVD", color="maroon")

# plt.plot(dims, rsvd_times, label="Randomized SVD")
plt.xlabel("Dimension", fontsize=fsize)
plt.ylabel("Time (s)", fontsize=fsize)
plt.xticks(fontsize=ticksize)
plt.yticks(fontsize=ticksize)
# plt.title("Computation Time vs. Dimension", fontsize=fsize)
plt.legend(fontsize=ticksize)
# plt.yscale('log')

### Make plots of estimates

The randomized SVD performs well in this case, but it cannot handle complex numbers. In the real data, this becomes and issue. Adding to the computational cost it seems better to focus more on the QR decomposition approach.

In [ ]:
qr_approx = np.sort(np.sum(np.absolute(R @ R.transpose()), axis=0))[::-1]
qr_approx = qr_approx / np.sum(np.absolute(qr_approx))

# qr_approx = np.sort(np.sum(R, axis=1))[::-1]
# qr_approx = qr_approx/np.sum(np.absolute(qr_approx))

# qr_approx = np.sort(np.diagonal(R@R.transpose()))[::-1]/np.sum(np.absolute(np.diagonal(R@R.transpose())))

# qr_approx = np.sort(np.diagonal(R))[::-1]/np.sum(np.absolute(np.diagonal(R)))

actual_eigenval = np.sort(eigenvals)[::-1]
actual_eigenval = actual_eigenval / np.sum(actual_eigenval)
svd_approx = S**2
svd_approx = svd_approx / np.sum(svd_approx)
rsvd_approx = rS**2
rsvd_approx = rsvd_approx / np.sum(rsvd_approx)

plt.plot(actual_eigenval, "g-o", label="Actual eigen decomp")
plt.plot(qr_approx, "b-s", label="QR approx")
plt.plot(svd_approx, "r-*", label="SVD approx")
plt.plot(rsvd_approx, "y-*", label="Randomized SVD approx")

plt.xlabel("Descending Order", fontsize=fsize)
plt.ylabel("Normalised Eignenvalue", fontsize=fsize)
plt.title("Eigenvalue Decay", fontsize=fsize)

plt.legend(fontsize=fsize)

# plt.plot(np.sort(np.diagonal(R@R.transpose()))[::-1]/np.sum(np.absolute(np.diagonal(R@R.transpose()))))
# plt.plot(S/np.sum(S))

# plt.plot(np.sort(np.diagonal(R))[::-1])
# plt.plot(S)

### Different approximations with QR

Consider different ways of approximating the eigenvalue decomposition of the coherence matrix with QR decomposition and plot them.

In [ ]:
qr_approx = np.sort(np.sum(np.absolute(R @ R.transpose()), axis=0))[::-1]
qr_approx = qr_approx / np.sum(np.absolute(qr_approx))

qr_approx1 = np.sort(np.absolute(np.sum(R, axis=1)))[::-1]
qr_approx1 = qr_approx1 / np.sum(np.absolute(qr_approx1))

qr_approx2 = np.sort(np.absolute(np.diagonal(R @ R.transpose())))[
    ::-1
] / np.sum(np.absolute(np.diagonal(R @ R.transpose())))

qr_approx3 = np.sort(np.absolute(np.diagonal(R)))[::-1] / np.sum(
    np.absolute(np.diagonal(R))
)

actual_eigenval = np.sort(eigenvals)[::-1]
actual_eigenval = actual_eigenval / np.sum(actual_eigenval)

plt.plot(actual_eigenval, "g-o", label="Actual eigen decomp")
plt.plot(qr_approx, "r-*", label="QR approx 1")
plt.plot(qr_approx1, "k-s", label="QR approx 2")
plt.plot(qr_approx2, "c-s", label="QR approx 3")
plt.plot(qr_approx3, "y-s", label="QR approx 4")
plt.plot(svd_approx, "k--*", label="SVD approx")


plt.xlabel("Descending Order", fontsize=fsize)
plt.ylabel("Normalised Eignenvalue", fontsize=fsize)
plt.title("Eigenvalue Decay", fontsize=fsize)
plt.legend(fontsize=fsize)

### R@Rh check

In the end, we choose to go with the approximation based on the diagonal of R@Rh where R is from the QR decomposition of the coherence matrix. Summing the rows of R also seems to work well and might be worth exploring more in the future. Now we look at R@R.transpose() to see how close it is to a diagonal matrix.

In [ ]:
RRh = R @ R.transpose()

plt.imshow(
    RRh, vmin=0, vmax=np.percentile(np.absolute(RRh), 50), aspect="auto"
)

plt.colorbar()

Plot gershgorin circles R@R.transpose() to see how well diagonal constrains the eigenvalues. Since all the discs are overlapping, we cannot conclude anything about the eigenvalues.

In [ ]:
# for a in range(len(RRh)):
for a in range(20):
    plt.plot(
        [a, a, a],
        [RRh[a, a] - RRh[a, :].sum(), RRh[a, a], RRh[a, a] + RRh[a, :].sum()],
        "o-",
    )

## Try with real data

We now try the considered tests with real data from Brady's Hot Spring.

In [ ]:
data_dir = Path(r"D:\CSM\Mines_Research\Test_data\Brady_Hotspring")

file = data_dir / "PoroTomo_iDAS16043_160314083818.h5"
data, _ = loadBradyHShdf5(file, normalize="no")

file = data_dir / "PoroTomo_iDAS16043_160314083848.h5"
data2, _ = loadBradyHShdf5(file, normalize="no")

# file = r"D:\CSM\Mines_Research\Test_data\Brady_Hotspring\PoroTomo_iDAS16043_160314083918.h5"
# data3,_= loadBradyHShdf5(file,normalize='no')

data = np.append(data, data2, axis=1)
# data = np.append(data,data3[:,:5000],axis=1)

samples_per_sec = 1000

In [ ]:
# start_ch = 1000
# nchannels = 3000
start_ch = 3100
nchannels = 2000
nsensors = 200
norm_win_spectra, frequencies = normalised_windowed_spectra(
    data[start_ch : nchannels + start_ch : int(nchannels / nsensors)],
    5,
    2.5,
    sample_interval=0.001,
)

# welch_coherence = np.matmul(norm_win_spectra.transpose(2,1,0), np.conjugate(norm_win_spectra.transpose(2,0,1)))
welch_coherence_mat = np.matmul(
    norm_win_spectra, np.conjugate(norm_win_spectra.transpose(0, 2, 1))
)
welch_coherence_mat = np.absolute(welch_coherence_mat) ** 2

Timing test for approximating eigenvalues with verious methods using real data.

In [ ]:
RandA = norm_win_spectra[300, :, :]
nreps = 10

approx_rank = int(min(RandA.shape) / 2)
approx_rank = min(RandA.shape)
approx_rank = int(len(RandA) / 2)

t0 = time.time()
for i in range(nreps):
    Q, R = np.linalg.qr(RandA)
    # qr_approx2 = np.sum(np.absolute(R@R.transpose()), axis=0)**2
    qr_approx = np.diag(np.absolute(R @ R.transpose())) ** 2

    Q2, R2 = np.linalg.qr(RandA.T)
    qr_approx2 = np.diag(np.absolute(R2 @ R2.transpose())) ** 2
t1 = time.time()
qr_time = t1 - t0

print("QR time: ", qr_time)

t0 = time.time()
for i in range(nreps):
    U, S, Vh = np.linalg.svd(RandA)
    svd_approx = S**4
t1 = time.time()
svd_time = t1 - t0

print("SVD time: ", svd_time)

t0 = time.time()
for i in range(nreps):
    coherence_mat = np.absolute(RandA @ np.conjugate(RandA.transpose())) ** 2
    eigenvals, _ = np.linalg.eig(coherence_mat)
t1 = time.time()
eig_time = t1 - t0

print("Eigenvalue time: ", eig_time)

t0 = time.time()
for i in range(nreps):
    rU, rS, rVh = randomized_svd(abs(RandA), approx_rank)
    rsvd_approx = rS**2
t1 = time.time()
rsvd_time = t1 - t0

print("Randomized SVD time: ", rsvd_time)

In [ ]:
fsize = 12
qr_approx = np.sum(np.absolute(R @ R.transpose()), axis=0)
qr_approx = np.sort(qr_approx)[::-1]
# qr_approx = np.sort(np.sum(R@R.transpose(), axis=0))[::-1]
qr_approx = qr_approx / np.sum(np.absolute(qr_approx))

actual_eigenval = np.sort(eigenvals)[::-1]
actual_eigenval = actual_eigenval / np.sum(actual_eigenval)
svd_approx = S**2
svd_approx = svd_approx / np.sum(svd_approx)
rsvd_approx = rS**2
rsvd_approx = rsvd_approx / np.sum(rsvd_approx)

plt.plot(
    actual_eigenval[: 6 * len(qr_approx)], "g-o", label="Actual eigen decomp"
)
plt.plot(qr_approx, "b-s", label="QR approx")
plt.plot(svd_approx, "r-*", label="SVD approx")
plt.plot(rsvd_approx, "y-*", label="Randomized SVD approx")

plt.xlabel("Descending Order", fontsize=fsize)
plt.ylabel("Normalised Eignenvalue", fontsize=fsize)
plt.title("Eigenvalue Decay", fontsize=fsize)
# plt.yscale("log")
plt.legend(fontsize=fsize)

In [ ]:
# np.max(norm_win_spectra)
# np.max(welch_coherence_mat.imag)
# np.max(coherence2)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(welch_coherence_mat[35, :, :].real)
plt.colorbar()
plt.subplot(1, 2, 2)
plt.imshow(welch_coherence_mat[35, :, :].imag)
plt.colorbar()

In [ ]:
# Create two 3D arrays (example data)
num_frames = int(len(welch_coherence_mat) / 5)  # 30
data1 = welch_coherence_mat[:].real  # Replace with your 3D data array 1

# Create a function to update the animations
fig = plt.figure(figsize=(6, 6))


def update(frame):
    fig.clear()
    frame *= 5
    # Update the first animation
    ax = fig.add_subplot(111)
    ax.imshow(
        data1[frame, :, :],
        cmap=plt.cm.viridis,
        animated=True,
        extent=[0, 1, 0, 1],
    )  # ,  vmin=-np.percentile(abs(data1[frame]),90), vmax=np.percentile(abs(data1[frame]),90))
    # ax.imshow(data1[frame], cmap=plt.cm.viridis, animated=True,  vmin=-epsilon, vmax=epsilon)
    # ax.imshow(data1[frame], cmap=plt.cm.PuOr, animated=True)
    # ax.imshow(data1[frame], interpolation="bicubic", cmap=plt.cm.RdBu, animated=True)
    ax.set_title("Frequency = " + str(frequencies[frame]), size=16)

    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

    plt.tight_layout()


# Create the animations
fig = plt.figure(figsize=(6, 6))
ani = FuncAnimation(fig, update, frames=num_frames, repeat=False)

# Display the animations side by side
display(HTML(ani.to_jshtml()))

In [ ]:
fsize = 15
# num_frames = coherence2.shape[0]
# data_2use = welch_coherence_mat.real
data_2use = np.absolute(welch_coherence_mat) ** 2
num_frames = int(data_2use.shape[0] / 2)

eig_ratios2 = np.empty(num_frames)
eig_ratios_qr = np.empty(num_frames)
eig_ratios_qr2 = np.empty(num_frames)
eig_ratios_svd = np.empty(num_frames)
eig_ratios_rsvd = np.empty(num_frames)

for d in range(num_frames):
    eigenvals, _ = np.linalg.eig(data_2use[d * 2])
    eigenvals = np.sort(eigenvals)[::-1]
    eig_ratios2[d] = eigenvals[0] / np.sum(eigenvals)

    Q, R = np.linalg.qr(norm_win_spectra[d * 2])
    qr_approx = np.sort(np.sum(np.absolute(R @ R.transpose()), axis=0))[::-1]
    # qr_approx = np.sort(np.sum(R@R.transpose(), axis=0))[::-1]
    eig_ratios_qr[d] = qr_approx[0] / np.sum(np.absolute(qr_approx))

    Q, R = np.linalg.qr(norm_win_spectra[d * 2].T)
    qr_approx2 = np.sort(np.diag(np.absolute(R @ R.transpose())))[::-1]
    # qr_approx = np.sort(np.sum(np.absolute(R@R.transpose()), axis=0))[::-1]
    # qr_approx = np.sort(np.sum(R@R.transpose(), axis=0))[::-1]
    eig_ratios_qr2[d] = qr_approx2[0] / np.sum(np.absolute(qr_approx2))

    U, S, Vh = np.linalg.svd(norm_win_spectra[d * 2])
    svd_approx = S**2
    eig_ratios_svd[d] = svd_approx[0] / np.sum(svd_approx)

    # rU, rS, rVh = randomized_svd(norm_win_spectra[d*2], approx_rank)
    # rsvd_approx = rS**2
    # eig_ratios_rsvd[d] = rsvd_approx[0]/np.sum(rsvd_approx)

In [ ]:
plt.plot(frequencies[:num_frames], eig_ratios2, label="Actual")
plt.plot(frequencies[:num_frames], eig_ratios_qr, label="QR", alpha=0.8)
plt.plot(frequencies[:num_frames], eig_ratios_svd, label="SVD", alpha=0.4)
# plt.plot(frequencies[:num_frames], eig_ratios_rsvd, label="rSVD",alpha=0.2)
plt.ylabel(r"$\frac{\lambda_1}{\sum_{i=1}^{n}{\lambda_i}}$", fontsize=fsize)
plt.xlabel("Frequency", fontsize=fsize)
plt.title(
    "Proportion of $\lambda_1$ in sum of all eigenvalues", fontsize=fsize
)
plt.xticks(fontsize=fsize)
plt.yticks(fontsize=fsize)
plt.legend(fontsize=fsize)

In [ ]:
RRh = (R @ np.conjugate(R.transpose())).real

plt.imshow(
    RRh, vmin=0, vmax=np.percentile(np.absolute(RRh), 90), aspect="auto"
)

plt.colorbar()

Plot gershgorin circles around RRh estimates to see overlap

In [ ]:
num_eigs = len(RRh)
gershgorin_upper = np.empty(num_eigs)
gershgorin_lower = np.empty(num_eigs)
for a in range(num_eigs):
    gershgorin_upper[a] = np.abs(RRh[a, :]).sum()
    gershgorin_lower[a] = max(2 * RRh[a, a] - np.abs(RRh[a, :]).sum(), 0)

ind = np.argsort(np.diagonal(RRh))[::-1]
plt.plot(gershgorin_upper[ind], "r-*", label="Upper bound")
plt.plot(gershgorin_lower[ind], "b-*", label="Lower bound")
plt.plot(np.diagonal(RRh)[ind], "g-*", label="Diagonal")